In [1]:
import numpy as np
import pandas as pd
import yaml

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

# ------------------------------
# Load settings and initialize
# ------------------------------
with open('../../Settings.yaml', 'r') as file:
    Setting = yaml.safe_load(file)

# ------------------------------
# Load energy related datasets
# ------------------------------
file_name = 'Unadjusted.xlsx'
file_path = f"{Setting['Output_Path_Unajusted']}/{file_name}"

Energy_Price = pd.read_excel(file_path, sheet_name='Input_Energy_By_Activity')
Energy_Usage = pd.read_excel(file_path, sheet_name='Energy_By_Activity')

# Use just sub-industry category 
Energy_Price = Energy_Price[Energy_Price['Industry_Category_Code'] == 2]
Energy_Price.drop(columns='Industry_Name',inplace=True)

# ------------------------------
# Load PPI datasets
# ------------------------------
file_name_1 = 'General.xlsx'
file_path = f"{Setting['Output_Path_General']}/{file_name_1}"

PPI = pd.read_excel(file_path, sheet_name='PPI')
PPI = PPI[PPI['Industry_Category_Code'] == 2]
PPI.drop(columns={'Industry_Category_Code','Industry_Name'},inplace=True)
PPI["Year"] = pd.to_numeric(PPI["Year"], errors="coerce").astype("Int64")
PPI["Industry_Code"] = pd.to_numeric(PPI["Industry_Code"], errors="coerce").astype("Int64")

# ------------------------------
# Harmonize merge keys (fix object vs int merge issue)
# ------------------------------
keys = ["Year", "Industry_Code"]

for d in (Energy_Price, Energy_Usage):
    d["Year"] = pd.to_numeric(d["Year"], errors="coerce").astype("Int64")
    d["Industry_Code"] = pd.to_numeric(d["Industry_Code"], errors="coerce").astype("Int64")

# ------------------------------
# Merge
# ------------------------------

df = Energy_Price.merge(Energy_Usage, on=keys, how="inner")

# ------------------------------
# Build coal+charcoal cost aggregate
# ------------------------------

df["Coal_Charcoal"] = df[["Coal", "Charcoal"]].sum(axis=1, skipna=True)

# ------------------------------
# Map cost columns -> usage columns
# ------------------------------
map_cols = {
    "Natural_Gas": "Natural_Gas (M3)",
    "Fueloil": "Fueloil (Liter)",
    "Gasoil": "Gasoil (Liter)",
    "Gasoline": "Gasoline (Liter)",
    "Kerosene": "Kerosene (Liter)",
    "LNG": "LNG (KG)",
    "Electricity": "Electricity (KWH)",
    "Coal_Charcoal": "Charcoal and Coal (KG)",
    "Water":"Water (M3)"
}

# ------------------------------
# Compute unit prices: cost / usage in Milion Rial
# ------------------------------
for cost_col, use_col in map_cols.items():
    if cost_col not in df.columns:
        df[cost_col] = np.nan
    if use_col not in df.columns:
        df[use_col] = np.nan

    den = pd.to_numeric(df[use_col], errors="coerce")
    num = pd.to_numeric(df[cost_col], errors="coerce")

    df[f"{cost_col}_unit_price"] = np.where(den > 0, (num / den), np.nan)

# ------------------------------
# Final output nominal
# ------------------------------
unit_price_cols = [c for c in df.columns if c.endswith("_unit_price")]
energy_unit_price = df[keys + ["Industry_Name"] + unit_price_cols]
energy_unit_price.replace(0, np.nan,inplace=True)

# ------------------------------
# Final output real
# ------------------------------
energy_real_unit_price = pd.merge(energy_unit_price,PPI,how='inner',on=keys)
energy_real_unit_price[unit_price_cols] = 100 * energy_real_unit_price[unit_price_cols].div(
    energy_real_unit_price['Price_Index'], axis=0)
energy_real_unit_price.drop(columns='Price_Index',inplace=True)

# ------------------------------
#Export nominal Energy Rate to Excel (in Milion Rial)
# ------------------------------
output_file_name = 'Unadjusted.xlsx'
sheet_name= 'Energy_Nominal_Rate'
path = f"{Setting['Output_Path_Unajusted']}/{output_file_name}"
with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    energy_unit_price.to_excel(writer,sheet_name=sheet_name ,index=False)

# ------------------------------
#Export Real Energy Rate to Excel (in Milion Rial)
# ------------------------------
output_file_name = 'Unadjusted.xlsx'
sheet_name= 'Energy_Real_Rate'
path = f"{Setting['Output_Path_Unajusted']}/{output_file_name}"
with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    energy_real_unit_price.to_excel(writer,sheet_name=sheet_name ,index=False)

C:\Users\hwi\AppData\Local\Temp\ipykernel_3824\1339299127.py:94: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  energy_unit_price.replace(0, np.nan,inplace=True)
